# ROC Curve

Generate the ROC Curve for each trained model

In [ ]:
# Reproducibility settings
import numpy as np
from sklearn.utils import check_random_state

SEED = 12345

# The NumPy Generator will be used throughout the whole experiment
# rng = np.random.default_rng(SEED)
np.random.seed(SEED)
rng = check_random_state(SEED)

In [ ]:
# Model Selection and Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import RocCurveDisplay

# Model Persistence
from joblib import load

# Visualisation
from matplotlib import pyplot as plt

import pandas as pd
import os

import warnings

warnings.filterwarnings("ignore")

In [ ]:
dataset_csv = os.getenv("DATA")
df = pd.read_csv(dataset_csv)

In [ ]:
X, y = df[df.columns[df.columns != "Class"]], df["Class"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=rng)

In [ ]:
import os

dt_near_miss = os.getenv("DECISION_TREE_NMISS")
rf_near_miss = os.getenv("RANDOM_FOREST_NMISS")

dt_smote = os.getenv("DECISION_TREE_SMOTE")
rf_smote = os.getenv("RANDOM_FOREST_SMOTE")

In [ ]:
# Gather all the model persistence files (.joblib) and sort them alphabetically

NEAR_MISS_KEY = "under_sampling_near_miss"
SMOTE_KEY = "over_sampling_smote"

model_files_per_sampler = {
    NEAR_MISS_KEY: [dt_near_miss, rf_near_miss],
    SMOTE_KEY: [dt_smote, rf_smote],
}

In [ ]:
def extract_name_string(model_file_path):
    filename, _ = os.path.splitext(os.path.basename(model_file_path))
    filename = filename.replace("gs_", "")
    return " ".join([s.title() for s in filename.split("_")])

In [ ]:
figure = plt.figure()
for sampler in model_files_per_sampler:
    for model_file in model_files_per_sampler[sampler]:
        print(f"loading: {model_file}")
        gs_model = load(model_file)
        name_string = extract_name_string(model_file)
        RocCurveDisplay.from_estimator(gs_model, X_test, y_test, ax=plt.gca(), name=name_string)
plt.show()